# Chromatin simulation with MiChroM — SBBf 2026

First tutorial: initial structure → collapse → production → analysis. No manual file uploads are required.

1. Save your own copy in Google Colab.
2. Select **Runtime → Change runtime type → GPU**.
3. Run the cells in order. If Colab requests a session restart after package installation, restart and run again from the beginning.
4. Set `SAVE_TO_DRIVE=True` to preserve results in Drive, or download the ZIP archive at the end. Choose a new `RUN_NAME` for another simulation.

This exercise retains the original **230-bead** model. The experimental files contain **1,356 bins of 100 kb** and are shown as a reference for the full chromosome. Quantitative comparison requires confirming genomic alignment and the segment annotations; the two maps are not treated as directly equivalent matrices.

The sign of PC1 is arbitrary: assigning A/B compartments requires additional biological information. The plot retains the convention of the source material. The simulation uses the subcompartment sequence provided below.

The original durations are retained: 40,000 collapse steps and 120,000 production steps. This is a teaching exercise; a short trajectory does not demonstrate equilibrium or convergence.

## 1. Installation and environment

OpenMiChroM 1.1.1 and OpenMM 8.6.1 (CUDA 12), as in the FI tutorial.

In [ ]:
# Install before importing scientific packages. Run on a fresh Colab runtime.
import sys
import subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'openmm[cuda12]==8.6.1', 'OpenMiChroM==1.1.1',
                           'numpy', 'pandas', 'matplotlib', 'scikit-learn', 'h5py', 'mdtraj', 'imageio', 'pillow'])
else:
    print('Local run: use your existing OpenMM/OpenMiChroM environment.')

In [ ]:
import os
from pathlib import Path
from importlib.metadata import version
import openmm as mm
from openmm import unit

SAVE_TO_DRIVE = False
RUN_NAME = 'chr10_workshop_01'  # Change for a fresh, independent run.
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in ('.', '..')
if IN_COLAB:
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        base = Path('/content/drive/MyDrive/sbbf2026')
    else:
        base = Path('/content/sbbf2026')
    WORKDIR = base / RUN_NAME
else:
    WORKDIR = Path.cwd()
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
Path('input').mkdir(exist_ok=True)
Path('output').mkdir(exist_ok=True)

SIM_PLATFORM = 'CUDA' if IN_COLAB else 'CPU'
platform = mm.Platform.getPlatformByName(SIM_PLATFORM)
system = mm.System()
system.addParticle(1.0)
integrator = mm.VerletIntegrator(0.001)
properties = {'DeviceIndex': '0', 'Precision': 'mixed'} if SIM_PLATFORM == 'CUDA' else {}
try:
    context = mm.Context(system, integrator, platform, properties)
    context.setPositions([[0, 0, 0]])
    integrator.step(1)
    print('OpenMM device check passed:', context.getPlatform().getName())
finally:
    if 'context' in globals(): del context
    del integrator
print('Working directory:', Path.cwd())
for package in ['OpenMM', 'OpenMiChroM', 'numpy', 'pandas']:
    print(package, version(package))
# Avoid mixing trajectories or images from different runs.
if Path('output_chr10/initialStats.txt').exists() or list(Path('output_chr10').glob('*.cndb')):
    raise RuntimeError('This folder already contains a simulation. Choose a new RUN_NAME and rerun from the beginning.')

In [ ]:
from urllib.request import urlopen
import io
import numpy as np
base_url = 'https://raw.githubusercontent.com/cagliarijv/Minicurso_simulacao_cromatina/principal/'
for name in ['chr10_100k.dense', 'chr10_100k.eigen']:
    if not Path(name).exists():
        with urlopen(base_url + name, timeout=60) as response:
            payload = response.read()
        values = np.loadtxt(io.BytesIO(payload))
        assert np.isfinite(values[~np.isnan(values)]).all(), f'Invalid data: {name}'
        Path(name).write_bytes(payload)
    print('Input available:', name)

In [ ]:
# Required libraries
from sklearn.preprocessing import normalize
from OpenMiChroM.ChromDynamics import MiChroM
from OpenMiChroM.CndbTools import cndbTools
from matplotlib.cm import ScalarMappable
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib as mpl
cndbTools = cndbTools()
from openmm import unit
import mdtraj as md
import pandas as pd
import numpy as np
import imageio.v2 as imageio
import os

print('Libraries imported successfully')

## 2. Experimental reference data

In [ ]:
# Figure dimensions (adjust as needed)
fig_width = 7  # Figure width
fig_height = 7  # Figure height

# Load the Hi-C file
#filename = '/content/chr10_100k.dense'
filename = 'chr10_100k.dense'
hic_file = np.loadtxt(filename)

r = np.triu(hic_file, k=1)  # Keep the upper triangle
r[np.isnan(r)] = 0.0  # Replace NaN with zero
r = normalize(r, axis=1, norm='max')  # Normalize values
rd = np.transpose(r)  # Transpose to restore symmetry
r = r + rd + np.diag(np.ones(len(r)))  # Add a unit diagonal

print("Number of experimental bins: ", len(r))

# Create a custom colormap (blue → white → red)
vmax = r.max()
vmin = 0.00005

colors = [[0, "blue"],  # Minimum
          [0.5, "white"],  # Midpoint
          [1, "red"]]  # Maximum

colors = [[0, "white"],  # Minimum
          [1, "red"]]  # Maximum

cmap = mpl.colors.LinearSegmentedColormap.from_list("custom_cmap", colors)

# Create a figure with the specified dimensions
fig = plt.figure(figsize=(fig_width, fig_height))
ax = fig.add_subplot(111)  # Add axes

# Plot the matrix using the selected color scale
im = ax.matshow(r, norm=mpl.colors.LogNorm(vmin=vmin, vmax=vmax), cmap=cmap)
plt.colorbar(im, label="Contact frequency")

plt.title("Experimental Hi-C — full chromosome (reference)")
plt.show()

In [ ]:
# Load the .eigen file
# eigen = np.loadtxt('/content/chr10_100k.eigen')
eigen = np.loadtxt('chr10_100k.eigen')

# Process the data
index = np.arange(eigen.size)
df = pd.Series(eigen, index=index)

A1_df = pd.Series(np.zeros(eigen.size), index=index)
A1_df[df < 0] = df[df < 0]
B1_df = pd.Series(np.zeros(eigen.size), index=index)
B1_df[df > 0] = df[df > 0]


# Create the plot
fig, axes = plt.subplots(ncols=1, nrows=2, figsize=(6.4 * 3 / 2, 4.8 * 2 / 3), gridspec_kw={'height_ratios': [1, 10]})
fig.tight_layout()
fig.subplots_adjust(hspace=0.05)

# Sequence
A11_df = A1_df.copy()
A11_df[A1_df < 0] = 1
B11_df = B1_df.copy()
B11_df[B1_df > 0] = 1

axes[0].axis('off')
width = 1
axes[0].bar(A11_df.index, A11_df, width, color="tab:red")
axes[0].bar(B11_df.index, B11_df, width, color="tab:blue")
axes[0].set_xlim([-50, eigen.size + 50])

# Eigen PC1
step = 1
axes[1].fill_between(A1_df.index[::step], A1_df[::step], 0, facecolor="tab:red", alpha=0.8, label="A")
axes[1].fill_between(B1_df.index[::step], B1_df[::step], 0, facecolor="tab:blue", alpha=0.8, label="B")

axes[1].set_ylim([-0.05, 0.05])
axes[1].set_xlim([-50, eigen.size + 50])
axes[1].set_ylabel('Eigen values, PC1')
axes[1].set_xlabel('Genomic bin index (100 kb per bin)')
axes[1].legend(title='compartments', bbox_to_anchor=(1., 1.45), frameon=False)

## 3. Model and interactions

In [ ]:
# Initialize the chromosome 10 simulation
sim_chr10 = MiChroM(name="chr10", temperature=1.0, timeStep=0.01)

In [ ]:
# CUDA in Colab; CPU for local execution.
sim_chr10.setup(platform=SIM_PLATFORM, gpu='0', integrator='langevin',
                precision='mixed', deviceIndex='0')

In [ ]:
# Set the simulation output folder
sim_chr10.saveFolder('output_chr10')
print('Output folder output_chr10 created; see the file panel.')

In [ ]:
# Create the sequence file and write its contents
with open('chr10_beads.txt', "w") as file:
        file.write("""1 NA\n2 NA\n3 B1\n4 B1\n5 B1\n6 B1\n7 B1\n8 B1\n9 B1\n10 B1
    11 B1\n12 B1\n13 B1\n14 B1\n15 B1\n16 B1\n17 B1\n18 B1\n19 A1\n20 A1
    21 A1\n22 A1\n23 B1\n24 B1\n25 B1\n26 B1\n27 B1\n28 B1\n29 B1\n30 B1
    31 B1\n32 B1\n33 B1\n34 B1\n35 B1\n36 B1\n37 B2\n38 B2\n39 B2\n40 B2
    41 B2\n42 B2\n43 B2\n44 B2\n45 B2\n46 B2\n47 B2\n48 B2\n49 B2\n50 B2
    51 B2\n52 B2\n53 B2\n54 B2\n55 B2\n56 B2\n57 B2\n58 B2\n59 B2\n60 B2
    61 B1\n62 B1\n63 A2\n64 A2\n65 B1\n66 B1\n67 B2\n68 B2\n69 B2\n70 B2
    71 B1\n72 B1\n73 B2\n74 B2\n75 B1\n76 B1\n77 A2\n78 A2\n79 A2\n80 A2
    81 A2\n82 A2\n83 B1\n84 B1\n85 B2\n86 B2\n87 B2\n88 B2\n89 B2\n90 B2
    91 B2\n92 B2\n93 B2\n94 B2\n95 B2\n96 B2\n97 B2\n98 B2\n99 B2\n100 B2
    101 B2\n102 B2\n103 B2\n104 B2\n105 B2\n106 B2\n107 B2\n108 B2\n109 B1\n110 B1
    111 B1\n112 B1\n113 A2\n114 A2\n115 A2\n116 A2\n117 A1\n118 A1\n119 A1\n120 A1
    121 A2\n122 A2\n123 A2\n124 A2\n125 A2\n126 A2\n127 A2\n128 A2\n129 A2\n130 A2
    131 B1\n132 B1\n133 B1\n134 B1\n135 B1\n136 B1\n137 B2\n138 B2\n139 B1\n140 B1
    141 B2\n142 B2\n143 B2\n144 B2\n145 A2\n146 A2\n147 A2\n148 A2\n149 A2\n150 A2
    151 A2\n152 A2\n153 B2\n154 B2\n155 B2\n156 B2\n157 A2\n158 A2\n159 B2\n160 B2
    161 B2\n162 B2\n163 B2\n164 B2\n165 B2\n166 B2\n167 B2\n168 B2\n169 B2\n170 B2
    171 B2\n172 B2\n173 B2\n174 B2\n175 B2\n176 B2\n177 B2\n178 B2\n179 B2\n180 B2
    181 B2\n182 B2\n183 B2\n184 B2\n185 B2\n186 B2\n187 B2\n188 B2\n189 B2\n190 B2
    191 B2\n192 B2\n193 B2\n194 B2\n195 B2\n196 B2\n197 B2\n198 B2\n199 B2\n200 B2
    201 B2\n202 B2\n203 B2\n204 B2\n205 B2\n206 B2\n207 B2\n208 B2\n209 B2\n210 B2
    211 B2\n212 B2\n213 B2\n214 B2\n215 B2\n216 B2\n217 B2\n218 B2\n219 B2\n220 B2
    221 B2\n222 B2\n223 B2\n224 B2\n225 A2\n226 A2\n227 A2\n228 A2\n229 A2\n230 A2""")

print("Sequence file 'chr10_beads.txt' created successfully!")
sequence = np.loadtxt('chr10_beads.txt', dtype=str)
bead_types = sequence[:, 1]
print(f'Model: {len(bead_types)} beads; experimental reference: {hic_file.shape[0]} bins.')
assert len(bead_types) <= hic_file.shape[0]

In [ ]:
# Set the sequence file path
#file_path = '/content/chr10_beads.txt'
file_path = 'chr10_beads.txt'

# Create an initial random-walk structure from the sequence file
chr10 = sim_chr10.createRandomWalk(ChromSeq=file_path)

In [ ]:
# Load the chromatin structure into the simulator
sim_chr10.loadStructure(chr10, center=True)

In [ ]:
# Add forces and potentials to the system
sim_chr10.addFENEBonds(kFb=30.0)
sim_chr10.addAngles(kA=2.0)
sim_chr10.addRepulsiveSoftCore(eCut=4.0)
sim_chr10.addFlatBottomHarmonic()
sim_chr10.addTypetoType(mu=3.22, rc=1.78)
sim_chr10.addIdealChromosome(mu=3.22, rc=1.78, dinit=3, dend=200)

## 4. Initial configuration

In [ ]:
# Extract coordinate data
data = chr10

# Unpack the coordinates for plotting
x, y, z = zip(*data)

# Create a colormap
cmap = plt.cm.viridis  # Choose your preferred colormap

# Create the 3D figure and axes
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# Color the points by bead index
colors = cmap(np.linspace(0, 1, len(x)))  # Generate one color per point

# Draw the scatter plot
p = ax.scatter(x, y, z, c=colors, marker='.', s=80)

ax.plot(x, y, z, color='gray', linewidth=1)


# ax.set_xlim(-15, 15)
# ax.set_ylim(-15, 15)
# ax.set_zlim(-15, 15)

sm = ScalarMappable(norm=mpl.colors.Normalize(0, len(x)-1), cmap=cmap)
sm.set_array([])
fig.colorbar(sm, ax=ax, label='Index')

# Set labels, limits, and title
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Visualization of 3D Coordinates - Initial Configuration')

# Adjust the viewing angle (optional)
ax.view_init(elev=25, azim=-60)

# Show the plot
plt.show()
# Lightweight images for Colab. Simulation dynamics are unchanged.
FRAME_DPI = 100
FRAME_STRIDE = 3

def save_frame(positions, title, path, half_width=None):
    positions = np.asarray(positions)
    positions = positions - positions.mean(axis=0)
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection='3d')
    points = ax.scatter(*positions.T, c=np.arange(len(positions)), cmap='viridis',
                        vmin=0, vmax=len(positions)-1, s=18)
    ax.plot(*positions.T, color='gray', lw=0.7)
    if half_width is None:
        half_width = max(float(np.abs(positions).max()) * 1.1, 1)
    for setter in [ax.set_xlim, ax.set_ylim, ax.set_zlim]: setter(-half_width, half_width)
    ax.set_box_aspect((1, 1, 1))
    ax.set(xlabel='X (σ)', ylabel='Y (σ)', zlabel='Z (σ)', title=title)
    fig.colorbar(points, ax=ax, label='Bead index')
    fig.savefig(path, dpi=FRAME_DPI)
    plt.close(fig)

## 5. Collapse

In [ ]:
# Initial collapse: 40,000 steps (same duration as the original tutorial).
import time
block_colapso = 10**3
n_blocks_colapso = 40
sim_chr10.createSimulation()
actual = sim_chr10.simulation.context.getPlatform().getName()
if IN_COLAB and actual != 'CUDA':
    raise RuntimeError(f'The simulation selected {actual}, but CUDA was expected.')
output_dir = Path('gif_colapso')
output_dir.mkdir(exist_ok=True)
t0 = time.time()
initial_width = max(float(np.abs(chr10 - chr10.mean(axis=0)).max()) * 1.1, 1)
for i in range(n_blocks_colapso):
    sim_chr10.run(nsteps=block_colapso, report=False)
    positions = np.asarray(sim_chr10.getPositions())
    assert np.isfinite(positions).all(), 'Non-finite coordinates during collapse.'
    save_frame(positions, f'Collapse — block {i+1}', output_dir / f'frame_{i:03d}.png', initial_width)
    if (i+1) % 10 == 0: print(f'Collapse: {i+1}/{n_blocks_colapso} blocks')
print(f'Collapse completed in {(time.time()-t0)/60:.1f} min.')

In [ ]:
# Animate frames in chronological order (without reverse playback).
image_files = sorted(Path('gif_colapso').glob('frame_*.png'))
assert image_files, 'Run the preceding simulation stage first.'
with imageio.get_writer('animacao_colapso.gif', mode='I', duration=200, loop=0) as writer:
    for image_path in image_files:
        writer.append_data(imageio.imread(image_path))
display(Image(filename='animacao_colapso.gif', width=600))

## 6. Production

In [ ]:
# Production: 120,000 steps. Save one sample every 1,000 steps.
sim_chr10.removeFlatBottomHarmonic()
block = 10**3
n_blocks = 120
output_dir = Path('gif_simulacao')
output_dir.mkdir(exist_ok=True)
sim_chr10.createReporters(statistics=True, traj=True, trajFormat='cndb',
                         outputName='chr10', interval=block)
t0 = time.time()
for i in range(n_blocks):
    sim_chr10.run(nsteps=block, report=False)
    if i % FRAME_STRIDE == 0:
        positions = np.asarray(sim_chr10.getPositions())
        assert np.isfinite(positions).all(), 'Non-finite coordinates during production.'
        save_frame(positions, f'Production — block {i+1}', output_dir / f'frame_{i:03d}.png', initial_width)
    if (i+1) % 20 == 0: print(f'Production: {i+1}/{n_blocks} blocks')
# Close reporter files before reading or exporting results.
for reporter in sim_chr10.simulation.reporters:
    for value in vars(reporter).values():
        if hasattr(value, 'flush'): value.flush()
        if hasattr(value, 'close'): value.close()
sim_chr10.simulation.reporters.clear()
print(f'Production completed in {(time.time()-t0)/60:.1f} min.')

In [ ]:
# Animate frames in chronological order (without reverse playback).
image_files = sorted(Path('gif_simulacao').glob('frame_*.png'))
assert image_files, 'Run the preceding simulation stage first.'
with imageio.get_writer('animacao.gif', mode='I', duration=200, loop=0) as writer:
    for image_path in image_files:
        writer.append_data(imageio.imread(image_path))
display(Image(filename='animacao.gif', width=600))

## 7. Trajectory analysis

In [ ]:
# Read existing numeric frame keys directly to avoid ambiguity in frame selection.
import h5py
with h5py.File('output_chr10/chr10_0.cndb', 'r') as trajectory:
    frame_keys = sorted((key for key in trajectory if key.isdigit()), key=int)
    assert len(frame_keys) >= 2, 'At least two frames are required.'
    chr10_xyz = np.stack([trajectory[key][:] for key in frame_keys])
assert chr10_xyz.shape[1:] == (len(bead_types), 3)
assert np.isfinite(chr10_xyz).all()
print(f'Trajectory: {chr10_xyz.shape[0]} frames, {chr10_xyz.shape[1]} beads.')
chr10_sim_HiC = cndbTools.traj2HiC(chr10_xyz, mu=3.22, rc=1.78)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(chr10_sim_HiC, norm=mpl.colors.LogNorm(vmin=0.00005, vmax=1), cmap='Reds')
fig.colorbar(im, ax=ax, label='Contact probability')
ax.set(title='Simulated Hi-C — 230-bead segment', xlabel='Bead index', ylabel='Bead index')
plt.show()

In [ ]:
# Plot the radius of gyration
# Open the data file
# file_path = "/content/output_chr10/statistics.txt"
file_path = "output_chr10/statistics.txt"

# Load the data, treating # as the comment marker
data = np.atleast_1d(np.genfromtxt(file_path, delimiter=',', comments='#', names=True))

coluna = 'RG'
# Extract simulation steps and the selected observable
steps = data['Step']
temperature = data[coluna]

# Create the plot
plt.figure(figsize=(12, 5))
plt.plot(steps, temperature, linestyle='-', color='blue')
plt.xlabel('Step', fontsize=12)
plt.ylabel(coluna, fontsize=12)
plt.title(coluna, fontsize=14)
plt.grid(True)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Plot the selected simulation observable
coluna = 'Temperature'
# Extract simulation steps and the selected observable
steps = data['Step']
temperature = data[coluna]

# Create the plot
plt.figure(figsize=(12, 5))
plt.plot(steps, temperature, linestyle='-', color='blue')
plt.xlabel('Step', fontsize=12)
plt.ylabel(coluna, fontsize=12)
plt.title(coluna, fontsize=14)
plt.grid(True)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Plot the selected simulation observable
coluna = 'Etotal'
# Extract simulation steps and the selected observable
steps = data['Step']
temperature = data[coluna]

# Create the plot
plt.figure(figsize=(12, 5))
plt.plot(steps, temperature, linestyle='-', color='blue')
plt.xlabel('Step', fontsize=12)
plt.ylabel(coluna, fontsize=12)
plt.title(coluna, fontsize=14)
plt.grid(True)
plt.tight_layout()

# Show the plot
plt.show()

## 8. Organization by type and export

In [ ]:
# Radial density by type, measured from the center of the whole chromosome.
# Avoid applying global indices to coordinates already filtered by type.
centered_xyz = chr10_xyz - chr10_xyz.mean(axis=1, keepdims=True)
radii = np.linalg.norm(centered_xyz, axis=2)
edges = np.linspace(0, max(float(radii.max()), 1e-6) * 1.001, 41)
radial_centers = 0.5 * (edges[:-1] + edges[1:])
shell_volume = (4 * np.pi / 3) * np.diff(edges**3)
type_xyz = {}
fig, ax = plt.subplots(figsize=(9, 4))
for label, color in [('A1','red'), ('A2','purple'), ('B1','blue'), ('B2','green'), ('NA','gray')]:
    selection = bead_types == label
    if not selection.any(): continue
    type_xyz[label] = centered_xyz[:, selection, :]
    counts, _ = np.histogram(radii[:, selection], bins=edges)
    density = counts / (len(chr10_xyz) * selection.sum() * shell_volume)
    ax.plot(radial_centers, density, label=label, color=color)
ax.set(xlabel='Distance from center (σ)', ylabel='Density per bead (σ⁻³)', title='Radial distribution by type')
ax.legend(); plt.show()

In [ ]:
# Consistent export: CNDB stores numerical coordinates in units of σ.
# For visualization only, use 1 σ = 1 nm; this is not a biological calibration.
topology = md.Topology()
chain = topology.add_chain()
conversion = {'A1':'ASP', 'A2':'GLU', 'B1':'LYS', 'B2':'ARG', 'NA':'GLY'}
previous = None
for label in bead_types:
    residue = topology.add_residue(conversion.get(label, 'GLY'), chain)
    atom = topology.add_atom('CA', md.element.carbon, residue)
    if previous is not None: topology.add_bond(previous, atom)
    previous = atom
traj = md.Trajectory(chr10_xyz.astype(np.float32), topology,
                     time=np.arange(len(chr10_xyz)) * block * sim_chr10.timeStep)
traj[0].save_pdb('output_chr10/chr10_0.pdb')
traj.save_xtc('output_chr10/chr10_0.xtc')
print('PDB and XTC exported. Numerical time uses τ as a visualization convention.')

In [ ]:
# Mean-squared displacement after removing center-of-mass translation.
# Short trajectory: do not automatically interpret this as a diffusion coefficient.
max_lag = max(1, len(chr10_xyz) // 2)
lags = np.arange(1, max_lag + 1)
fig, ax = plt.subplots(figsize=(8, 4))
for label, coords in type_xyz.items():
    msd = [np.mean(np.sum((coords[lag:] - coords[:-lag])**2, axis=2)) for lag in lags]
    ax.loglog(lags * block * sim_chr10.timeStep, msd, label=label)
ax.set(xlabel='Time lag (τ)', ylabel='MSD (σ²)', title='MSD by type — center-of-mass translation removed')
ax.legend(); plt.show()

## 9. Download results

Save the notebook too, to preserve plots and comments. The ZIP archive contains the run files, including trajectories, statistics, and animations.

In [ ]:
import shutil
archive = shutil.make_archive(str(Path.cwd().parent / (Path.cwd().name + '_results')), 'zip', root_dir=Path.cwd())
print('Results archive:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)